# Home Workouts Chatbot - Trainer Build

**Built for a workout coach assistant:**
- Streamlined home workout data ingestion
- Conversation memory for follow-up coaching
- Parent document retrieval for better exercise context

**NEW IN v5 Build:**
- **Multi-Query Retrieval**: Cover more workout question variations
- **FlashRank Reranking**: Provide more relevant exercise answers

## Warmup: Install Dependencies

In [ ]:
%pip install "langchain==0.3.27" -qqq
%pip install "langchain-community==0.3.31" -qqq
%pip install "langchain-openai==0.3.35" -qqq
%pip install "langchain-chroma==0.2.6" -qqq
%pip install -U cryptography -qqq
%pip install pypdf -qqq
%pip install gradio -qqq
#--------------------------------------------------------------------------------
# NEW IN v5 Build: Install FlashRank for reranking
#--------------------------------------------------------------------------------
%pip install flashrank -qqq

## Setup Configuration

## Load Trainer Secrets


In [2]:
def load_secrets(keys):
    """Load secret values quickly with minimal imports."""
    import os
    from typing import Dict

    # Fast-path: if all keys already in environment, return immediately
    env_values: Dict[str, str] = {k: os.getenv(k) for k in keys}
    if all(env_values.get(k) for k in keys):
        return env_values

    # Only attempt to load .env if a .env file exists and some keys are missing
    missing_initial = [k for k, v in values.items() if not v]
    env_path = os.path.join(os.getcwd(), ".env")
    if missing_initial and os.path.exists(env_path):
        try:
            from dotenv import load_dotenv
            load_dotenv(env_path)
        except Exception as e:
            raise ImportError("python-dotenv is required for local development. Install it with: pip install python-dotenv") from e
        # Re-read environment after loading .env
        values = {k: os.getenv(k) for k in keys}
    else:
        # Use whatever is present in the environment
        values = {k: os.getenv(k) for k in keys}

    # Check for missing keys and provide helpful error for local development
    missing = [k for k, v in values.items() if not v]
    if missing: 
        env_file_help = (
            f"Missing keys: {', '.join(missing)}.\n\n"
            "Create a .env file in the project root with:\n"
            + "\n".join([f"{key}=YOUR_VALUE_HERE" for key in missing])
        )
        raise ValueError(env_file_help)

    # Only set variables that are not already present to avoid overwriting runtime config
    for key, value in values.items():
        if value and os.getenv(key) is None:
            os.environ[key] = value

    return values

In [3]:
try:
    secrets = load_secrets(['CHROMA_API_KEY', 'CHROMA_TENANT', 'CHROMA_DATABASE', 'OPENAI_API_KEY'])
    print ("Secrets loaded successfully:")
    print (f"CHROMA_API_KEY: {secrets['CHROMA_API_KEY'][:4]}...{secrets['CHROMA_API_KEY'][-4:]}")
    print (f"CHROMA_TENANT: {secrets['CHROMA_TENANT'][:4]}...{secrets['CHROMA_TENANT'][-4:]}")
    print (f"CHROMA_DATABASE: {secrets['CHROMA_DATABASE'][:4]}...{secrets['CHROMA_DATABASE'][-4:]}")
    print (f"OPENAI_API_KEY: {secrets['OPENAI_API_KEY'][:4]}...{secrets['OPENAI_API_KEY'][-4:]}")
except ValueError as e:
    print(e)


Secrets loaded successfully:
CHROMA_API_KEY: ck-8...3Tei
CHROMA_TENANT: 562a...449d
CHROMA_DATABASE: Proj...ject
OPENAI_API_KEY: sk-p...VaUA


## Workout Engine Variables

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector Database collection name
COLLECTION_NAME = f"home_workout_database"

# Embeddings Model
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini"
)

# Classification LLM
classification_llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

# Vector Database
vectorstore = Chroma(
  embedding_function=embeddings,
  collection_name=COLLECTION_NAME,
  chroma_cloud_api_key=secrets["CHROMA_API_KEY"],
  tenant=secrets["CHROMA_TENANT"],
  database=secrets["CHROMA_DATABASE"]
)

# Base retriever from vectorstore
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## Workout Document Ingestion

In [5]:
#@title Clean Text Function

import re

def clean_text(text: str) -> str:
    """
    TEXT PREPROCESSING - Clean PDF

    Args:
        text (str): Raw PDF text

    Returns:
        str: Cleaned text
    """
    # Remove multiple whitespaces
    text = re.sub(r'\s+', ' ', text)

    # Remove standalone page numbers
    text = re.sub(r'(?<=\.)\s*\d+\s*$', '', text)

    # Remove whitespace from beginning and end
    text = text.strip()

    return text

In [6]:
#@title Topic Detection Function

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def detect_document_topic(documents: list) -> str:
    """
    DETECT DOCUMENT TOPIC

    Args:
        documents (list): List of Document objects from PyPDFLoader

    Returns:
        str: Detected topic (e.g., 'workout exercises', 'home exercises')
    """

    topic_detection_template = ChatPromptTemplate.from_template(
      """
      Analyze the following document content and determine its primary topic.

      Document content:
      {content}

      Based on this content, what is the primary topic? Answer with a single word or short phrase (e.g., 'home exercises', 'workout').

      Examples:
      If the document is about physical exercises, answer: physical workout

      Primary topic:
      """
    )

    topic_detection_chain = topic_detection_template | classification_llm | StrOutputParser()

    # Extract sample content from first few pages
    sample_content = ""
    for doc in documents[:3]:
        sample_content += doc.page_content + "\n\n"

    # Limit to 4000 characters to save costs
    sample_content = sample_content[:4000]

    detected_topic = topic_detection_chain.invoke({
        "content": sample_content
    }).strip().lower()

    return detected_topic

In [7]:
#@title Ingest Function

from langchain.document_loaders import PyPDFLoader

def ingest_documents(document_path: str):
    """
    INGESTION PIPELINE

    Args:
        document_path (str): URL or local path to PDF

    Returns:
        tuple: (num_pages, detected_topic)
    """

    print("-" * 80)
    print("STARTING INGESTION PIPELINE")
    print("-" * 80)

    #--------------------------------------------------------------------------------
    # STEP 1: LOAD DOCUMENTS
    #--------------------------------------------------------------------------------
    print("\n[1/6] Loading file...")

    loader = PyPDFLoader(document_path)
    documents = loader.load()

    print(f"✓ Loaded {len(documents)} pages from file")

    #--------------------------------------------------------------------------------
    # STEP 2: AUTO-DETECT TOPIC
    #--------------------------------------------------------------------------------
    print(f"\n[2/6] Auto-detecting document topic...")

    detected_topic = detect_document_topic(documents)

    print(f"✓ Topic auto-detected: '{detected_topic}'")

    #--------------------------------------------------------------------------------
    # STEP 3: PREPROCESSING
    #--------------------------------------------------------------------------------
    print(f"\n[3/6] Applying text preprocessing...")

    for doc in documents:
      doc.page_content = clean_text(doc.page_content)

    print(f"✓ Cleaned {len(documents)} pages")

    #--------------------------------------------------------------------------------
    # STEP 4: CHUNKING
    #--------------------------------------------------------------------------------
    print(f"\n[4/6] Splitting documents into chunks...")

    text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=2000,
      chunk_overlap=200
    )

    chunks = text_splitter.split_documents(documents)

    print(f"✓ Split {len(chunks)} chunks")

    #--------------------------------------------------------------------------------
    # STEP 5: ADD METADATA
    #--------------------------------------------------------------------------------
    print(f"\n[5/6] Enriching documents with metadata...")

    for chunk in chunks:
      chunk.metadata.update({
          'topic': detected_topic,
      })

    print(f"✓ Metadata enriched for all documents")

    #--------------------------------------------------------------------------------
    # STEP 6: Use Base Retriever (NEW IN v5 Build)
    #--------------------------------------------------------------------------------
    print(f"\n[6/6] ParentDocumentRetriever processing...")

    base_retriever.add_documents(chunks)
    
    print(f"✓ Stored")

    return len(chunks), detected_topic

In [8]:
#ingest_documents("https://bitcoin.org/bitcoin.pdf")

ingest_documents("files/100-workouts-vol1.pdf")
ingest_documents("files/100-workouts-vol2.pdf")
ingest_documents("files/100-workouts-vol3.pdf")
ingest_documents("files/100-workouts-vol4.pdf")


--------------------------------------------------------------------------------
STARTING INGESTION PIPELINE
--------------------------------------------------------------------------------

[1/6] Loading file...
✓ Loaded 205 pages from file

[2/6] Auto-detecting document topic...
✓ Topic auto-detected: 'bodyweight training'

[3/6] Applying text preprocessing...
✓ Cleaned 205 pages

[4/6] Splitting documents into chunks...
✓ Split 103 chunks

[5/6] Enriching documents with metadata...
✓ Metadata enriched for all documents

[6/6] ParentDocumentRetriever processing...
✓ Stored
--------------------------------------------------------------------------------
STARTING INGESTION PIPELINE
--------------------------------------------------------------------------------

[1/6] Loading file...
✓ Loaded 207 pages from file

[2/6] Auto-detecting document topic...
✓ Topic auto-detected: 'workout routines'

[3/6] Applying text preprocessing...
✓ Cleaned 207 pages

[4/6] Splitting documents into chun

Could not reliably determine page label for 1.
Could not reliably determine page label for 2.
Could not reliably determine page label for 3.
Could not reliably determine page label for 4.
Could not reliably determine page label for 5.
Could not reliably determine page label for 6.
Could not reliably determine page label for 7.
Could not reliably determine page label for 8.
Could not reliably determine page label for 9.
Could not reliably determine page label for 10.
Could not reliably determine page label for 11.
Could not reliably determine page label for 12.
Could not reliably determine page label for 13.
Could not reliably determine page label for 14.
Could not reliably determine page label for 15.
Could not reliably determine page label for 16.
Could not reliably determine page label for 17.
Could not reliably determine page label for 18.
Could not reliably determine page label for 19.
Could not reliably determine page label for 20.
Could not reliably determine page label for 21.
C

✓ Loaded 207 pages from file

[2/6] Auto-detecting document topic...
✓ Topic auto-detected: 'bodyweight workouts'

[3/6] Applying text preprocessing...
✓ Cleaned 207 pages

[4/6] Splitting documents into chunks...
✓ Split 105 chunks

[5/6] Enriching documents with metadata...
✓ Metadata enriched for all documents

[6/6] ParentDocumentRetriever processing...
✓ Stored


(105, 'bodyweight workouts')

## Chat Inference

In [9]:
#@title RAG Chain Creation

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain():
    """
    CREATE RAG CHAIN

    Returns:
        RunnableSequence: RAG chain
    """

    rag_template = ChatPromptTemplate.from_template(
      """
      You are a personal trainer answering questions about home workout exercises.

      Use the conversation history and provided documents to answer the current question.
      If the question references previous context (e.g., "it", "this", "that"), use the conversation history to understand what is being referenced.

      Instructions:
      - Answer clearly and concisely (max 2-3 paragraphs)
      - Use only the provided documents as context
      - If the question is a follow-up, maintain conversation continuity
      - If the answer is not in the documents, say "I don't have enough information to answer that question"

      Conversation History:
      {history}

      Documents:
      {context}

      Current Question: {query}

      Answer:
      """
    )

    rag_chain = rag_template | llm | StrOutputParser()

    return rag_chain

In [10]:
#@title Format Chat History Helper (NEW IN v4 Build)

def format_chat_history(history: list, max_turns: int = 5) -> str:
    """
    FORMAT CHAT HISTORY

    Converts Gradio's messages format history into a readable string.

    WHY LIMIT TURNS?
    - Token limits: Long conversations exceed context window
    - Relevance: Recent turns are more relevant
    - Cost: Fewer tokens = lower cost
    - Performance: Faster processing

    Args:
        history (list): Gradio chat history in messages format
        max_turns (int): Maximum number of conversation turns to include

    Returns:
        str: Formatted conversation history
    """

    if not history:
        return "No previous conversation."

    # Limit to last N turns (each turn = user + assistant message)
    recent_history = history[-(max_turns * 2):]

    # Format as readable conversation
    formatted = []
    for message in recent_history:
        role = message["role"].capitalize()
        content = message["content"]
        formatted.append(f"{role}: {content}")

    return "\n".join(formatted)

In [11]:
#@title Inference with MultiQuery + Reranking (NEW IN v5 Build)

from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

def inference(
    query: str,
    chat_history: list = None
) -> str:
    """
    INFERENCE WITH MULTI-QUERY RETRIEVAL + RERANKING 

    MULTI-QUERY:
    - LLM generates query variations
    - Each variation searches the vector store
    - Combines and deduplicates results

    RERANKING:
    - Cross-encoder scores each doc against ORIGINAL query
    - Reranks all documents by relevance
    - Keeps only top_n most relevant

    TRADE-OFFS:
    - Slower (LLM calls for query expansion + reranking)
    - More expensive (more tokens processed)
    - BUT: Maximum quality for complex/ambiguous queries
    """

    print("="*80)
    print("RUNNING MULTI-QUERY INFERENCE")
    print("="*80)

    #--------------------------------------------------------------------------------
    # STEP 1: FORMAT CONVERSATION HISTORY
    #--------------------------------------------------------------------------------
    print(f"\n[1/4] Formatting conversation history...")
    formatted_history = format_chat_history(chat_history, max_turns=5)
    print(f"✓ History formatted ({len(chat_history or [])} messages)")

    #--------------------------------------------------------------------------------
    # STEP 2: MULTI-QUERY RETRIEVAL + RERANKING
    #--------------------------------------------------------------------------------
    print(f"\n[2/4] Performing multi-query retrieval + reranking...")

    # Multi-query retriever: generates query variations
    multiquery_retriever = MultiQueryRetriever.from_llm(
      retriever=base_retriever,
      llm=llm
    )

    # Reranker: cross-encoder scoring
    compressor = FlashrankRerank(
        top_n=4
    )

    # Compression retriever: combines multiquery + reranking
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=multiquery_retriever
    )

    results = compression_retriever.invoke(query)

    print(f"✓ Retrieved and reranked {len(results)} documents")

    #--------------------------------------------------------------------------------
    # STEP 3: FORMAT CONTEXT
    #--------------------------------------------------------------------------------
    print(f"\n[3/4] Formatting context for LLM...")
    context = "\n\n".join([doc.page_content for doc in results])

    #--------------------------------------------------------------------------------
    # STEP 4: GENERATE ANSWER
    #--------------------------------------------------------------------------------
    print(f"\n[4/4] Invoking RAG chain...")

    response = create_rag_chain().invoke({
        "context": context,
        "query": query,
        "history": formatted_history
    })

    print("\n" + "="*80)
    print("INFERENCE COMPLETE")
    print("="*80)

    return response

In [12]:
# Test multi-query inference
inference("What exercises can I do at home?")

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...


RUNNING MULTI-QUERY INFERENCE

[1/4] Formatting conversation history...
✓ History formatted (0 messages)

[2/4] Performing multi-query retrieval + reranking...


ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [01:00<00:00, 1.70MiB/s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain.retrievers.multi_query:Generated queries: ['What are some effective home workouts I can try?  ', 'Can you suggest some exercises that I can easily do at home?  ', 'What type of home-based fitness routines are available for me?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.trychroma.com/api/v2/tenants/562ad7bd-8347-4865-9d4e-1ba9b093449d/databases/Project/collections/fb44e507-5d27-4392-bf4c-4a93b69d57f1/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.trychroma.com/api/v2/tenants/562ad7bd-8347-4865-9d4e-1ba9b093449d/databases/Project/collections/fb44e507-5d27-4392-bf4c-4a93b69d57f1/query "HTTP/1.1 200 OK"
INFO:httpx:HTT

✓ Retrieved and reranked 4 documents

[3/4] Formatting context for LLM...

[4/4] Invoking RAG chain...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



INFERENCE COMPLETE


"You can engage in various bodyweight exercises at home that require no equipment, making them accessible to anyone. These exercises can include push-ups, squats, lunges, planks, and burpees. These moves are designed to improve strength, tone muscle, and increase aerobic capacity, allowing you to work on your fitness without the need for additional equipment.\n\nIf you're focusing on specific areas, you can also incorporate exercises like pull-ups if you have access to a pull-up bar or monkey bars. For a structured approach, you can choose workouts labeled as High Burn or Strength oriented from a collection of workouts based on your fitness goals, ensuring a varied regimen that promotes overall conditioning. Remember to start with Level I workouts if you're new to bodyweight training and gradually increase the intensity as you become more comfortable."

## Trainer Demo

In [ ]:
def chat_inference(message, history):
  """Gradio ChatInterface wrapper"""
  return inference(query=message, chat_history=history)

In [14]:
import gradio as gr

demo = gr.ChatInterface(
    fn=chat_inference,
    title="ChatBot for Home Workouts",
    description="Ask questions about home workout exercises.",
    examples=[
        "What exercises can I do at home?",
        "How often should I workout?",
        "What are some good cardio exercises?",
        "How do I build muscle at home?"
    ],
)

demo.launch(share=True, debug=True)

c:\Users\rafag\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-initiated-analytics "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://api.gradio.app/v3/tunnel-request "HTTP/1.1 200 OK"



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-error-analytics "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/https%3A/api.gradio.app/gradio-launched-telemetry "HTTP/1.1 200 OK"


...


INFERENCE COMPLETE
RUNNING MULTI-QUERY INFERENCE

[1/4] Formatting conversation history...
✓ History formatted (6 messages)

[2/4] Performing multi-query retrieval + reranking...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain.retrievers.multi_query:Generated queries: ['What insights can you share regarding the topic of politics?  ', 'Can you provide an overview of key concepts and current issues in politics?  ', 'What are the main theories and systems of governance that relate to politics?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.trychroma.com/api/v2/tenants/562ad7bd-8347-4865-9d4e-1ba9b093449d/databases/Project/collections/fb44e507-5d27-4392-bf4c-4a93b69d57f1/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.trychroma.com/api/v2/tenants/562ad7bd-8347-4865-9d4e-1ba9b093449d/databases/Project/collections/fb44e507-5d27-4392-bf4c-4a93b69d57f1/query "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1

✓ Retrieved and reranked 4 documents

[3/4] Formatting context for LLM...

[4/4] Invoking RAG chain...


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



INFERENCE COMPLETE
Keyboard interruption in main thread... closing server.
